In [1]:
# Importing Libraries

import os
import numpy as np
import pandas as pd

import sklearn.linear_model
import sklearn.metrics

In [2]:
def remove_words_from_string(input_string, bad_words):
    """
    Removes words from a string that also appear in a list.

    Args:
      input_string: x_train element of text.
      word_list: stop words to filter out of.

    Returns:
      A new string with the specified words removed.
    """

    words = input_string.split()  # Split the string into a list of words
    new_string = []

    for word in words:
        if word not in bad_words:  # Check if the word is in the list
            new_string.append(word)  # Add the word to the new list

    return " ".join(new_string)  # Join the words back into a string
        

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split

def tune_hyperparameters_and_fit(x_train_dF, y_train_d):

    
#pipeline acts a blueprint for a logistic regression model
    pipeline = Pipeline([
        #('scaler', StandardScaler(with_mean=False)),                  # Feature scaling
        #('poly', PolynomialFeatures()),                # Optional polynomial features
       #('svd', TruncatedSVD(n_components=100)),
        ('logistic', LogisticRegression(max_iter=2000))  # Logistic regression classifier
    ])

    # parameter grid for hyperparameter optimizing
    #commented out most things because it was really slow.
    param_grid = {
        #'poly__degree': [1],
        # include a bias term in the polynomial features (interaction only)
        #'poly__interaction_only': [False, True],
        # PCA: try no dimensionality reduction or keep enough components to explain 95% variance
        #'pca__n_components': [None, 0.95],
        # penalty strength
#         'logistic__C': [0.01, 0.05, 0.1, 0.5, 1, 5, 10], #, 0.05, 0.1, 0.5, 1, 2, 5, 10
        'logistic__C': [0.1], #, 0.05, 0.1, 0.5, 1, 2, 5, 10
        # lasso or ridge penalty
        'logistic__penalty': ['l2','l1'], #'l1', 
        # Solvers that support both 'l1' and 'l2'; 'liblinear' is common for small datasets
        'logistic__solver': ['saga', 'sag','lbfgs'], #'saga'  #'lbfgs' doesnt work with l1
        # idk what this does tbh :D
        #'logistic__class_weight': [None, 'balanced']
    }

    # Create GridSearchCV object with 5-fold cross-validation
    lr_model_trained = GridSearchCV(pipeline, param_grid, cv=2, scoring='roc_auc', n_jobs=-1)
    

    # Fit grid search on training data
    lr_model_trained.fit(x_train_dF, y_train_d)

    print("\n\n\n\nOptimal model hyperparameters:")
    print(lr_model_trained.best_params_)

    return lr_model_trained


In [4]:

def contains_strings(arr):
    # Check if the dtype indicates a string
    if arr.dtype.kind in ('U', 'S'):
        return True
    # For object arrays, check each element
    if arr.dtype.kind == 'O':
        return any(isinstance(x, str) for x in arr.flatten())
    return False



from scipy.sparse import hstack
from scipy.sparse import csr_matrix
def create_sparse_training_set(x_unprocessed_df, bag_of_words):
    #d is the number of data points, z is number of columns from 4 onwards 
    nonBagOfWords_cols_dz = x_unprocessed_df.iloc[:, 4:].values
    #convert to sparse so that when stacked with bag_of_words (which is already sparse)
    #the end result remains sparse
    sparse_nonBagOfWords_cols_dz = csr_matrix(nonBagOfWords_cols_dz)
    x_train_dF = hstack([sparse_nonBagOfWords_cols_dz, bag_of_words])
    
    assert(x_train_dF.shape == (x_unprocessed_df.shape[0],nonBagOfWords_cols_dz.shape[1] + bag_of_words.shape[1]))

    return x_train_dF


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
import re

def predict_on_xtest_and_write_to_file(lr_model_trained, vectorizer, stopwords):

    data_dir = 'data_readinglevel'
    x_unprocessed_df = pd.read_csv(os.path.join(data_dir, 'x_test.csv'))


    #for bag of words
    tr_text_list = x_unprocessed_df['text'].values.tolist()

    # Make list / array to actually store elements
    element_list  = []
    for element in tr_text_list:
        element = re.sub('[^A-Za-z]', ' ', element) # keep only letters and spaces
        element = element.lower()                   # convert to lowercase
        element = remove_words_from_string(element, stopwords)
        element_list.append(element)
    
    new_bag_of_words = vectorizer.transform(element_list)

    x_train_dF = create_sparse_training_set(x_unprocessed_df, new_bag_of_words)
    
    predictions_d = lr_model_trained.predict_proba(x_train_dF)
    with open("yproba1_test.txt", "w") as file:
        for element in predictions_d:
            file.write(str(element[1]) + "\n")

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
import re
from sklearn.feature_extraction.text import TfidfVectorizer


if __name__ == '__main__':
    data_dir = 'data_readinglevel'
    x_unprocessed_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
    y_unprocessed_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))

    N, n_cols = x_unprocessed_df.shape
    print("Shape of x_unprocessed_df: (%d, %d)" % (N, n_cols))
    print("Shape of y_unprocessed_df: %s" % str(y_unprocessed_df.shape))

    tr_text_list = x_unprocessed_df['text'].values.tolist()

    # Make list of stopwords to filter text with
    with open("stopwords.txt", "r") as file:
        stopwords_list = file.read()
    stopwords = set(stopwords_list.splitlines())
    stopwords = list(stopwords)

    #for bag of words
    #vectorizer = CountVectorizer(min_df=0.01, max_df=0.8)     
    vectorizer = TfidfVectorizer()

    # Make list / array to actually store elements
    element_list  = []
    for element in tr_text_list:
        element = re.sub('[^A-Za-z]', ' ', element) # keep only letters and spaces
        element = element.lower()                   # convert to lowercase
        element = remove_words_from_string(element, stopwords)
        element_list.append(element)
    
    bag_of_words = vectorizer.fit_transform(element_list)
    

    assert(bag_of_words.shape[0] == x_unprocessed_df.shape[0])
    print(x_unprocessed_df.shape)
    #we shouldn't do  bag_of_words.toarray()
    #this turns bag_of_words from a sparse matrix into a dense numpy array
    #sklearn can directly use a sparse matrix, but idk how to do it yet
    #sparse matrix will hopefully make it run a lot faster because so far its hella slow
    #x_train_dF = np.column_stack((x_unprocessed_df.iloc[:, 4:].values, bag_of_words.toarray()))

    #this uses sparse matrix
    x_train_dF = create_sparse_training_set(x_unprocessed_df, bag_of_words)
    print("x_train_df shape: ")
    print(x_train_dF.shape)
    
 

    #takes the course labels column of unprocessed labels and turns each label into binary 1 or 0
    y_train_d = (y_unprocessed_df.iloc[:, 3].values.flatten() == "Key Stage 4-5").astype(int)

    lr_model_trained = tune_hyperparameters_and_fit(x_train_dF, y_train_d)

    accuracy = lr_model_trained.score(x_train_dF, y_train_d)
    print("\n\n\n\n\nTest Accuracy:", accuracy)
    

    predict_on_xtest_and_write_to_file(lr_model_trained, vectorizer=vectorizer, stopwords=stopwords)
    


    

Shape of x_unprocessed_df: (5557, 32)
Shape of y_unprocessed_df: (5557, 5)
(5557, 32)
x_train_df shape: 
(5557, 25254)


/Users/aviolet/micromamba/envs/cs135_25s_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/aviolet/micromamba/envs/cs135_25s_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/aviolet/micromamba/envs/cs135_25s_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/aviolet/micromamba/envs/cs135_25s_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/aviolet/micromamba/envs/cs135_25s_env/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:349: ConvergenceWarning: The max_iter was reached which means





Optimal model hyperparameters:
{'logistic__C': 0.1, 'logistic__penalty': 'l2', 'logistic__solver': 'lbfgs'}





Test Accuracy: 0.7936135947335001


In [7]:
# Somehow combine bag of words and other data (from part 1) with 
# representation in part 2
def create_sparse_training_set_P2(x_unprocessed_df, BERT):
    #d is the number of data points, z is number of columns from 4 onwards 
    nonBERT_cols_dz = x_unprocessed_df.iloc[:, 4:].values
    
    # convert into sparse matrix?
    sparse_nonBERT_cols_dz = csr_matrix(nonBERT_cols_dz)
    print("shape of nonBERT_dz: ", sparse_nonBERT_cols_dz.shape)
    print("shape of BERT data: ", BERT.shape)
    
    
    # combine bert with regular data
    x_train_dF = hstack([sparse_nonBERT_cols_dz, BERT])
    print("shape of combined: (Should be (5557, 796) cuz combined)", x_train_dF.shape)
    
    return x_train_dF
    

In [8]:
# MLP Attempt 1
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV
# from sklearn.model_selection import train_test_split <-- Don't think we need this now

def tune_hyperparameters_and_fit_MLP(x_train_dF, y_train_d):
    print("Making model")
    
    # Possible Improvements:
    # Try different number of hidden layers
    # Try changing alpha values
    param_grid = {
        # number of hidden layers
        # 'hidden_layer_sizes': [(10,), (25,), (50,), (100,), (150,), (200,)],
        'hidden_layer_sizes': [(10,)],
        
        # alpha value: strength of L2 Regularization Term
        # 'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1]
        'alpha': [1e-4]
    }
    
    clf = MLPClassifier(random_state=1, max_iter=300)
    # clf = MLPClassifierMLPClassifier(random_state=1, max_iter=300, early_stopping=True)
    grid_search = GridSearchCV(clf, param_grid, scoring='roc_auc', n_jobs=-1)
    grid_search.fit(x_train_dF, y_train_d)
    
    # For testing:
    # print("Least Loss: ", clf.best_loss_)
    # print("Best Validation Score: ", clf.best_validation_score_)
    print("Best Parameters:", grid_search.best_params_)
    print("Best Validation Score:", grid_search.best_score_)
    
    return grid_search.best_estimator_

In [13]:
''' Demo of how to load BERT embeddings for train data.

Will print out embeddings and associated labels
for the SAME 8 randomly chosen examples from load_train_data.py
'''

import numpy as np
import pandas as pd
import os
import textwrap
from sklearn.model_selection import train_test_split

def load_arr_from_npz(npz_path):
    ''' Load array from npz compressed file given path

    Returns
    -------
    arr : numpy ndarray
    '''
    npz_file_obj = np.load(npz_path)
    arr = npz_file_obj.f.arr_0.copy() # Rely on default name from np.savez
    npz_file_obj.close()
    return arr

if __name__ == '__main__':
    data_dir = 'data_readinglevel'
    x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
    y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
    
    # Load BERT embeddings as 2D numpy array
    # Each row corresponds to row of data frames above
    # Each col is one of the H=768 dimensions of BERT embedding    
    xBERT_train_NH = load_arr_from_npz(os.path.join(
        data_dir, 'x_train_BERT_embeddings.npz'))
    assert xBERT_train_NH.ndim == 2

    N, n_cols = x_train_df.shape
    N2, H = xBERT_train_NH.shape
    
    # Combine BERT representation with given data
    x_train_P2 = create_sparse_training_set_P2(x_train_df, xBERT_train_NH)
    
    # Splitting into 90% training and 10% testing
    x_train, x_test, y_train, y_test = train_test_split(x_train_P2, y_train_df, test_size=0.1, random_state=42)

    # Now you can proceed with training your model on x_train and evaluating on x_test
    print("Shape of x_train:", x_train.shape)
    print("Shape of x_test:", x_test.shape)
    print("Shape of y_train:", y_train.shape)
    print("Shape of y_test:", y_test.shape)
    
    print("Before making model")
    
    # Create Multi-layer Perceptron classifier
    CLF_model = tune_hyperparameters_and_fit_MLP(x_train_P2, y_train_df)
#     CLF_model = tune_hyperparameters_and_fit_MLP(x_train, y_train)
    accuracy = CLF_model.score(x_test, y_test)
    print("accuracy: ", accuracy)
    print("finished making model\n")
    
    

shape of nonBERT_dz:  (5557, 28)
shape of BERT data:  (5557, 768)
shape of combined: (Should be (5557, 796) cuz combined) (5557, 796)


ValueError: could not convert string to float: 'A. A. Milne'

In [ ]:
# #     For testing

# print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
# print("Shape of y_train_df: %s" % str(y_train_df.shape))
# print("Shape of xBERT_train_NH: %s" % str(xBERT_train_NH.shape))

# # Print out 8 random entries
# tr_text_list = x_train_df['text'].values.tolist()
# prng = np.random.RandomState(101)
# rows = prng.permutation(np.arange(y_train_df.shape[0]))
# for row_id in rows[:8]:
#     text = tr_text_list[row_id]
#     print("row %5d | %s BY %s | y = %s" % (
#         row_id,
#         y_train_df['title'].values[row_id],
#         y_train_df['author'].values[row_id],
#         y_train_df['Coarse Label'].values[row_id],
#         ))
#     # Pretty print text via textwrap library
#     line_list = textwrap.wrap(tr_text_list[row_id],
#         width=70,
#         initial_indent='  ',
#         subsequent_indent='  ')
#     print('\n'.join(line_list))
#     print("BERT embedding vector (size %d):" % H)
#     with np.printoptions(precision=2, edgeitems=4, threshold=50):
#         print(xBERT_train_NH[row_id])
#     print("")
